# 5 projets d'IA avec Python

Notebook de l'article [5 projets d'IA à réaliser rapidement avec Python](https://moncoachdata.com/blog/5-projets-ia-avec-python/) de MonCoachData.

L'API OpenAI est simulée en local : aucune clé n'est nécessaire.

## Installation

`pdfkit` a besoin du programme `wkhtmltopdf`.

In [ ]:
%pip install -q openai pymupdf markdown pdfkit scikit-learn gradio sentence-transformers youtube-transcript-api
!which wkhtmltopdf || (apt-get -qq update && apt-get -qq install -y wkhtmltopdf)

## Versions utilisées

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn
import openai
import pymupdf
import markdown
import gradio as gr

print("Python", sys.version.split()[0])
for module in (np, pd, sklearn, openai, pymupdf, markdown, gr):
    print(f"{module.__name__:<12}{module.__version__}")

## Faux serveur OpenAI

L'API OpenAI est simulée par un serveur local : aucune clé n'est nécessaire, et les réponses du « modèle » sont pré-écrites. Avec une vraie clé, tu n'as pas besoin de cette cellule.

In [ ]:
import json
import threading
import http.server

recu = {}   # dernière requête reçue par le faux serveur

REPONSES = {
    "cv": """# Alex Martin
Data analyst junior, orienté automatisation de rapports

## Expérience
- **Data analyst, Boutique en ligne (2023-2025)**
  - Automatisation de rapports hebdomadaires avec Python et pandas pour l'équipe marketing
  - Création de tableaux de bord de ventes e-commerce avec Power BI
  - Communication des résultats aux équipes métier

## Compétences
Python, SQL, automatisation de rapports, pandas, Power BI""",
    "resume": "Résumé simulé : la vidéo explique comment automatiser des rapports avec Python. Points clés : 1) extraire les données, 2) les nettoyer avec pandas, 3) générer le rapport automatiquement.",
    "rag": "Réponse simulée : d'après le contexte fourni, les ventes augmentent de façon régulière sur l'année, avec une accélération au quatrième trimestre.",
}

class FauxOpenAI(http.server.BaseHTTPRequestHandler):
    def do_POST(self):
        requete = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        recu.update(requete)
        question = requete["messages"][-1]["content"]
        cle = "cv" if "Markdown" in question else "resume" if "transcription" in question.lower() else "rag"
        corps = json.dumps({"id": "faux", "object": "chat.completion", "created": 0, "model": requete["model"],
                            "choices": [{"index": 0, "finish_reason": "stop", "message": {"role": "assistant", "content": REPONSES[cle]}}]}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(corps)

    def log_message(self, *args):
        pass

serveur = http.server.HTTPServer(("127.0.0.1", 8001), FauxOpenAI)
threading.Thread(target=serveur.serve_forever, daemon=True).start()
print("Faux serveur OpenAI prêt sur http://127.0.0.1:8001/v1")

## Projet #1 : adapter son CV

Un CV et une offre fictifs, puis le prompt.

In [ ]:
md_resume = """# Alex Martin
Data analyst junior

## Expérience
- **Data analyst, Boutique en ligne (2023-2025)**
  - Création de tableaux de bord de ventes avec Power BI
  - Nettoyage de données avec Python et pandas
  - Analyses hebdomadaires pour l'équipe marketing

## Compétences
Python, SQL, pandas, Power BI, Excel"""

job_description = """Nous recherchons un data analyst pour automatiser nos rapports.
Compétences attendues : Python, SQL, automatisation de rapports, communication avec les équipes métier.
Une expérience en e-commerce est un plus."""

prompt = f"""
J'ai un CV formaté en Markdown et une description de poste.
Adapte mon CV pour qu'il corresponde mieux aux exigences du poste, avec un ton professionnel.
Mets en avant les compétences et les expériences les plus pertinentes pour le poste.
Mon CV doit toujours refléter mes vraies qualifications : n'invente rien.

### Voici mon CV en Markdown :
{md_resume}

### Voici la description du poste :
{job_description}

Modifie le CV de la manière suivante :
- Utilise les mots-clés et les phrases de la description du poste.
- Ajuste les puces sous chaque rôle pour mettre en valeur les réalisations pertinentes.
- Maintiens la clarté, la concision et le professionnalisme.

Retourne le CV mis à jour au format Markdown.
"""

print(len(prompt), "caractères")
print(prompt[:160] + "…")

Appel d'API (réponse pré-écrite par le faux serveur). En vrai : `client = OpenAI()` avec la clé dans `OPENAI_API_KEY`.

In [ ]:
from openai import OpenAI

# En vrai : client = OpenAI()  (la clé est lue dans la variable d'environnement OPENAI_API_KEY)
client = OpenAI(api_key="cle-factice", base_url="http://127.0.0.1:8001/v1")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Tu es un assistant utile."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.25,
)

resume = response.choices[0].message.content
print("Modèle demandé :", recu["model"], "| température :", recu["temperature"])
print(resume)

Markdown → HTML → PDF.

In [ ]:
import markdown
import pdfkit
import pymupdf

html_cv = markdown.markdown(resume)
html_cv = f"<html><head><meta charset='utf-8'></head><body style='font-family: sans-serif'>{html_cv}</body></html>"

pdfkit.from_string(html_cv, "cv_adapte.pdf")

with pymupdf.open("cv_adapte.pdf") as pdf:
    print("Pages :", len(pdf))
    print(pdf[0].get_text()[:70].strip())

## Projet #2 : résumer une vidéo YouTube

Extraction de l'ID de la vidéo.

In [ ]:
import re

def extraire_id_video(url):
    """Renvoie l'ID de 11 caractères d'une vidéo YouTube, ou None."""
    motif = r"(?:v=|/)([0-9A-Za-z_-]{11})(?:[?&#]|$)"
    match = re.search(motif, url)
    return match.group(1) if match else None

urls = [
    "https://www.youtube.com/watch?v=AbC123_-xyZ",
    "https://www.youtube.com/watch?v=AbC123_-xyZ&t=42s",
    "https://youtu.be/AbC123_-xyZ",
    "https://www.youtube.com/shorts/AbC123_-xyZ?feature=share",
    "https://www.youtube.com/embed/AbC123_-xyZ",
    "https://exemple.com/une-page",
]
for url in urls:
    print(f"{str(extraire_id_video(url)):<13}{url}")

La vraie récupération se fait avec `YouTubeTranscriptApi().fetch(video_id, languages=["fr", "en"])` (nécessite d'accéder à YouTube).

In [ ]:
import inspect
from youtube_transcript_api import YouTubeTranscriptApi, FetchedTranscript, FetchedTranscriptSnippet

print(inspect.signature(YouTubeTranscriptApi.fetch))

Une transcription fabriquée, du même type que celle de l'API.

In [ ]:
# YouTube n'est pas joignable ici : on fabrique une transcription du même type que celle de l'API
transcript = FetchedTranscript(
    snippets=[
        FetchedTranscriptSnippet(text="Aujourd'hui, on automatise nos rapports avec Python.", start=0.0, duration=3.2),
        FetchedTranscriptSnippet(text="D'abord, on extrait les données.", start=3.2, duration=2.5),
        FetchedTranscriptSnippet(text="Ensuite, on les nettoie avec pandas.", start=5.7, duration=2.8),
    ],
    video_id="AbC123_-xyZ", language="Français", language_code="fr", is_generated=False,
)

transcript_text = "\n".join(snippet.text for snippet in transcript)
print(transcript_text)

Résumé (réponse pré-écrite).

In [ ]:
prompt_resume = f"""Voici la transcription d'une vidéo YouTube.
Résume-la en 3 points clés.

Transcription :
{transcript_text}"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_resume}],
)
print(response.choices[0].message.content)

## Projet #3 : organiser ses PDF

Six PDF dans un dossier en désordre.

In [ ]:
import os

# 6 « articles » dans un dossier en désordre, avec des noms qui ne disent rien
os.makedirs("bureau", exist_ok=True)

resumes = [
    "Nous entraînons un réseau convolutif pour classer des images de radiographies. Les couches de convolution extraient des motifs visuels, et l'augmentation d'images améliore la précision.",
    "Cet article présente une méthode de détection d'objets dans des images satellites. Le modèle repère les bâtiments et les routes grâce à des convolutions sur les images.",
    "Nous étudions le résumé automatique de textes avec un modèle de langage de type transformeur. Le modèle apprend à produire des phrases courtes à partir de longs textes.",
    "Cet article compare plusieurs modèles de langage pour classer le sentiment de textes d'avis clients. Les mots et les phrases sont convertis en vecteurs avant la classification.",
    "Nous prévoyons la consommation d'électricité à partir de séries temporelles horaires. La saisonnalité et la tendance de la série sont modélisées pour réduire l'erreur de prévision.",
    "Cet article prévoit les ventes hebdomadaires d'un magasin à partir de séries temporelles. Le modèle capture la tendance, la saisonnalité et l'effet des promotions sur la prévision.",
]
ordre = [4, 0, 2, 5, 1, 3]   # on mélange les sujets

for numero, i in enumerate(ordre, start=1):
    doc = pymupdf.open()
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 545, 400), "Résumé\n\n" + resumes[i], fontsize=12)
    doc.save(f"bureau/scan_{numero:03d}.pdf")
    doc.close()

print(sorted(os.listdir("bureau")))

Lecture des résumés avec PyMuPDF.

In [ ]:
fichiers = sorted(f for f in os.listdir("bureau") if f.endswith(".pdf"))

abstracts = []
for fichier in fichiers:
    with pymupdf.open(f"bureau/{fichier}") as pdf:
        texte = pdf[0].get_text().replace("Résumé", "").strip().replace("\n", " ")
    abstracts.append(texte)

print(len(abstracts), "résumés lus")
print(abstracts[0])

TF-IDF + K-Means (TF-IDF remplace les embeddings, voir la cellule suivante pour la vraie version).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# Ici, des vecteurs TF-IDF jouent le rôle des embeddings (voir le texte de l'article)
mots_vides = ["le", "la", "les", "des", "de", "du", "un", "une", "et", "en", "pour", "sont", "est", "par", "sur", "dans",
              "cet", "ces", "nous", "avec", "au", "aux", "qui", "que", "avant", "à", "partir", "grâce", "plusieurs"]
vectoriseur = TfidfVectorizer(stop_words=mots_vides)
vecteurs = vectoriseur.fit_transform(abstracts)
print("Matrice :", vecteurs.shape)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
groupes = kmeans.fit_predict(vecteurs)

df_pdf = pd.DataFrame({"fichier": fichiers, "groupe": groupes}).sort_values(["groupe", "fichier"])
df_pdf

Rangement dans des dossiers.

In [ ]:
import shutil

termes = np.array(vectoriseur.get_feature_names_out())

for groupe in sorted(set(groupes)):
    centre = kmeans.cluster_centers_[groupe]
    mots = termes[centre.argsort()[::-1][:2]]            # les 2 mots les plus typiques du groupe
    dossier = f"bureau/groupe_{groupe}_{'_'.join(mots)}"
    os.makedirs(dossier, exist_ok=True)
    for fichier in df_pdf.loc[df_pdf["groupe"] == groupe, "fichier"]:
        shutil.move(f"bureau/{fichier}", f"{dossier}/{fichier}")

for racine, dossiers, files in sorted(os.walk("bureau")):
    for f in sorted(files):
        print(os.path.join(racine, f))

## Version avec de vrais embeddings (nécessite Internet)

Cette cellule télécharge un modèle depuis Hugging Face : elle fonctionne sur Google Colab.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(abstracts)
print(embeddings.shape)

kmeans_st = KMeans(n_clusters=3, random_state=42, n_init=10)
print(kmeans_st.fit_predict(embeddings))

## Projet #4 : recherche multimodale

Un rapport de 3 pages avec un graphique.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# un graphique que l'on va placer dans le PDF
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.bar(["T1", "T2", "T3", "T4"], [120, 150, 135, 180], color="#3a55c8")
ax.set_title("Ventes par trimestre (k€)")
fig.savefig("graphique_ventes.png", dpi=100, bbox_inches="tight")
plt.close(fig)

paragraphes = [
    "Le rapport annuel présente les résultats commerciaux de l'entreprise. Les ventes augmentent de façon régulière sur l'année, avec une accélération au quatrième trimestre. " * 4,
    "Le deuxième chapitre décrit la répartition des ventes par trimestre. Le graphique ci-dessous montre la progression trimestrielle des ventes. " * 4,
    "Le dernier chapitre présente les perspectives. L'entreprise prévoit de recruter des data analysts et d'investir dans l'automatisation des rapports. " * 4,
]

doc = pymupdf.open()
for texte in paragraphes:
    page = doc.new_page()
    page.insert_textbox(pymupdf.Rect(50, 50, 545, 330), texte, fontsize=11)
doc[1].insert_image(pymupdf.Rect(150, 350, 450, 538), filename="graphique_ventes.png")
doc.save("rapport.pdf")
doc.close()
print("rapport.pdf créé :", len(pymupdf.open("rapport.pdf")), "pages")

Découpage en morceaux qui se chevauchent.

In [ ]:
def extraire_chunks(pdf_path, chunk_size, overlap_size):
    """Découpe chaque page en morceaux de chunk_size caractères qui se chevauchent."""
    assert overlap_size < chunk_size, "le chevauchement doit être plus petit que le morceau"
    chunks = []
    with pymupdf.open(pdf_path) as pdf:
        for num_page, page in enumerate(pdf, start=1):
            texte = page.get_text()
            debut = 0
            while debut < len(texte):
                fin = debut + chunk_size
                chunks.append({"fichier": pdf_path, "page": num_page, "texte": texte[debut:fin]})
                if fin >= len(texte):
                    break
                debut += chunk_size - overlap_size
    return chunks

chunks = extraire_chunks("rapport.pdf", chunk_size=300, overlap_size=60)
df_chunks = pd.DataFrame(chunks)

print(df_chunks.groupby("page").size().to_string())
for i in range(2):
    print(f"\nChunk {i + 1} (page {df_chunks.page[i]}) :")
    print(df_chunks.texte[i].replace("\n", " "))

Extraction des images.

In [ ]:
with pymupdf.open("rapport.pdf") as pdf:
    for num_page, page in enumerate(pdf, start=1):
        for image in page.get_images(full=True):
            infos = pdf.extract_image(image[0])
            nom = f"image_page{num_page}.{infos['ext']}"
            with open(nom, "wb") as f:
                f.write(infos["image"])
            print(f"page {num_page} : {nom}, {infos['width']}x{infos['height']} pixels")

Recherche par similarité cosinus (TF-IDF à la place d'un modèle d'embedding).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Ici aussi, TF-IDF remplace le modèle d'embedding
vectoriseur = TfidfVectorizer()
matrice = vectoriseur.fit_transform(df_chunks["texte"])

def rechercher(requete, k=3):
    scores = cosine_similarity(vectoriseur.transform([requete]), matrice)[0]
    meilleurs = scores.argsort()[::-1][:k]
    return df_chunks.iloc[meilleurs].assign(score=scores[meilleurs].round(3))[["page", "score", "texte"]]

resultats = rechercher("perspectives de recrutement et automatisation")
for _, ligne in resultats.iterrows():
    print(f"page {ligne.page} | score {ligne.score} | {ligne.texte[:60].strip()}…".replace("\n", " "))

## Projet #5 : questions/réponses (RAG)

In [ ]:
def repondre(question, k=3):
    morceaux = rechercher(question, k)
    contexte = "\n\n".join(f"[page {ligne.page}] {ligne.texte}" for _, ligne in morceaux.iterrows())
    prompt = f"""Réponds à la question uniquement à partir du contexte ci-dessous.
Si la réponse n'y figure pas, dis-le.

Contexte :
{contexte}

Question : {question}"""
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    return response.choices[0].message.content, sorted(set(morceaux["page"]))

reponse, pages = repondre("Comment évoluent les ventes ?")
print("Pages utilisées :", pages)
print(reponse)

Interface Gradio.

In [ ]:
import gradio as gr

def generate_response(message, history):
    reponse, pages = repondre(message["text"])
    return f"{reponse}\n\n(Sources : pages {pages})"

demo = gr.ChatInterface(
    fn=generate_response,
    examples=[{"text": "Comment évoluent les ventes ?", "files": []}],
    title="Questions sur mon rapport",
    multimodal=True,
)

demo.launch(prevent_thread_lock=True)   # dans un script classique : demo.launch()

On interroge l'application par programme.

In [ ]:
from gradio_client import Client

app = Client(demo.local_url, verbose=False)
print(app.predict({"text": "Comment évoluent les ventes ?", "files": []}, api_name="/generate_response"))
demo.close()